In this second part of the project, we move from Python into pure SQL.
The three tables built in Part 1 are loaded into an in-memory DuckDB database, and all analyses in this part are done using only SQL. There are several things I want to demonstrate here:
- Demographic profile of the user base
- Churn (across multiple dimensions)
- How and why people switch tiers
- How does engagement relate to conversions

In [1]:
%load_ext sql
%sql duckdb:///:memory:
%config SqlMagic.displaylimit = None

Connecting to 'duckdb:///:memory:'

displaylimit: Value None will be treated as 0 (no limit)

In [2]:
%%sql
CREATE TABLE users         AS SELECT * FROM read_parquet('df_users.parquet');
CREATE TABLE subscriptions AS SELECT * FROM read_parquet('df_subscription.parquet');
CREATE TABLE activity      AS SELECT * FROM read_parquet('df_daily.parquet');

Running query in 'duckdb:///:memory:'

Count


Now that we have them loaded, we can do a quick sanity check, to determine the rows in each of the databases

In [3]:
%%sql
SELECT 'users'         AS table_name, COUNT(*) AS row_count FROM users
UNION ALL
SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL
SELECT 'activity',      COUNT(*) FROM activity;

Running query in 'duckdb:///:memory:'

table_name,row_count
users,100000
subscriptions,187223
activity,4179509


In [4]:
%%sql
SELECT * FROM users LIMIT 3;

Running query in 'duckdb:///:memory:'

user_id,join_date,is_churned,churn_date,age,gender,sexual preference,sexual orientation,country,city,ethnicity,work_status
1,2023-04-13 00:00:00,0,None,26,Male,Prefers women,Heterosexual,Germany,Berlin,White/Caucasian,Employed
2,2024-03-11 00:00:00,0,None,38,Male,Prefers women,Heterosexual,Italy,Milan,White/Caucasian,Employed
3,2023-09-28 00:00:00,0,None,30,Male,Prefers women,Heterosexual,Spain,Madrid,White/Caucasian,Employed


In [5]:
%%sql
SELECT * FROM subscriptions LIMIT 3;

Running query in 'duckdb:///:memory:'

user_id,date,action,tier
1,2025-01-01 00:00:00,free,free
2,2025-01-01 00:00:00,free,free
3,2025-01-01 00:00:00,plus,plus


In [6]:
%%sql
SELECT * FROM activity LIMIT 3;

Running query in 'duckdb:///:memory:'

user_id,date,likes,dislikes,matches,messages_sent,messages_received
14,2025-01-01 00:00:00,110,29,5,6,5
17,2025-01-01 00:00:00,52,45,4,3,2
17,2025-01-01 00:00:00,49,40,11,11,10


Okay, so we see which data we have - the demographics (users), the current and new tiers (subscriptions), and the activity (activity).
Let's start with some demographics of our userbase

In [7]:
%%sql
-- Gender split with percentage share and average age per group
SELECT
    gender,
    COUNT(*)                                                      AS user_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_of_total,
    ROUND(AVG(age), 1)                                            AS avg_age,
    MIN(age)                                                      AS min_age,
    MAX(age)                                                      AS max_age
FROM users
GROUP BY gender
ORDER BY user_count DESC;

Running query in 'duckdb:///:memory:'

gender,user_count,pct_of_total,avg_age,min_age,max_age
Male,60036,60.0,29.4,18,65
Female,35068,35.1,29.5,18,65
Non-binary,4896,4.9,29.2,18,65


Here, we can see that most of the userbase are men, and that the overall age range is between 18 and 65, with the average being around 29. We can see this in more details if we have a look at age in bins

In [8]:
%%sql
-- Bucketing users into standard age groups
SELECT
    CASE
        WHEN age BETWEEN 18 AND 24 THEN '18–24'
        WHEN age BETWEEN 25 AND 34 THEN '25–34'
        WHEN age BETWEEN 35 AND 44 THEN '35–44'
        WHEN age BETWEEN 45 AND 54 THEN '45–54'
        ELSE '55+'
    END                                                           AS age_group,
    COUNT(*)                                                      AS users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_of_total
FROM users
GROUP BY age_group
ORDER BY age_group;

Running query in 'duckdb:///:memory:'

age_group,users,pct_of_total
18–24,32530,32.5
25–34,45068,45.1
35–44,16383,16.4
45–54,4522,4.5
55+,1497,1.5


So, the user base is mostly focused on the 25-34 age range, followed by 18-24, while users older than that amount to around 22%.
Next, we can see where the user base is located.

In [9]:
%%sql
-- Users per country, with a breakdown of how many cities they span
SELECT
    country,
    COUNT(*)                                                      AS total_users,
    COUNT(DISTINCT city)                                          AS n_cities,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_of_total
FROM users
GROUP BY country
ORDER BY total_users DESC;

Running query in 'duckdb:///:memory:'

country,total_users,n_cities,pct_of_total
United Kingdom,24982,4,25.0
France,20257,4,20.3
Germany,19817,4,19.8
Spain,14964,4,15.0
Italy,14921,4,14.9
Netherlands,5059,4,5.1


Here, again, we can see that our dataset generation was successful - we have an even breakdown of people per country.
Next we can see how many people of different work status there is in the userbase, and look that up by gender.

In [10]:
%%sql
-- Cross-tab of work status and gender using conditional aggregation
SELECT
    work_status,
    COUNT(*)                                                      AS total,
    SUM(CASE WHEN gender = 'Male'       THEN 1 ELSE 0 END)       AS male,
    SUM(CASE WHEN gender = 'Female'     THEN 1 ELSE 0 END)       AS female,
    SUM(CASE WHEN gender = 'Non-binary' THEN 1 ELSE 0 END)       AS non_binary
FROM users
GROUP BY work_status
ORDER BY total DESC;

Running query in 'duckdb:///:memory:'

work_status,total,male,female,non_binary
Employed,70191,42021,24745,3425
Student,20004,12048,6948,1008
Unemployed,9805,5967,3375,463


That seems sufficient for the demographics for now. Next, we can have a look at the churn.

In [11]:
%%sql
SELECT
    COUNT(*)                                                      AS total_users,
    SUM(is_churned)                                               AS churned,
    COUNT(*) - SUM(is_churned)                                    AS still_active,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 2)                 AS churn_rate_pct
FROM users;

Running query in 'duckdb:///:memory:'

total_users,churned,still_active,churn_rate_pct
100000,20032,79968,20.03


There is 20% of people who churned throughout the year - let's break that down in different ways.

In [12]:
%%sql
SELECT
    gender,
    COUNT(*)                                                      AS total,
    SUM(is_churned)                                               AS churned,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 2)                 AS churn_rate_pct
FROM users
GROUP BY gender
ORDER BY churn_rate_pct DESC;

Running query in 'duckdb:///:memory:'

gender,total,churned,churn_rate_pct
Male,60036,12045,20.06
Female,35068,7012,20.0
Non-binary,4896,975,19.91


In [13]:
%%sql
SELECT
    CASE
        WHEN age BETWEEN 18 AND 24 THEN '18–24'
        WHEN age BETWEEN 25 AND 34 THEN '25–34'
        WHEN age BETWEEN 35 AND 44 THEN '35–44'
        WHEN age BETWEEN 45 AND 54 THEN '45–54'
        ELSE '55+'
    END                                                           AS age_group,
    COUNT(*)                                                      AS total,
    SUM(is_churned)                                               AS churned,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 2)                 AS churn_rate_pct
FROM users
GROUP BY age_group
ORDER BY age_group;

Running query in 'duckdb:///:memory:'

age_group,total,churned,churn_rate_pct
18–24,32530,6469,19.89
25–34,45068,9054,20.09
35–44,16383,3301,20.15
45–54,4522,913,20.19
55+,1497,295,19.71


In [14]:
%%sql
SELECT
    country,
    COUNT(*)                                                      AS total,
    SUM(is_churned)                                               AS churned,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 2)                 AS churn_rate_pct
FROM users
GROUP BY country
ORDER BY churn_rate_pct DESC;

Running query in 'duckdb:///:memory:'

country,total,churned,churn_rate_pct
Netherlands,5059,1050,20.76
Germany,19817,4070,20.54
Italy,14921,3031,20.31
Spain,14964,2976,19.89
France,20257,3999,19.74
United Kingdom,24982,4906,19.64


In [15]:
%%sql
SELECT
    work_status,
    COUNT(*)                                                      AS total,
    SUM(is_churned)                                               AS churned,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 2)                 AS churn_rate_pct
FROM users
GROUP BY work_status
ORDER BY churn_rate_pct DESC;

Running query in 'duckdb:///:memory:'

work_status,total,churned,churn_rate_pct
Student,20004,4039,20.19
Unemployed,9805,1975,20.14
Employed,70191,14018,19.97


Next, we can have a look at the time to churn. If a user was using the app prior to 2025, we will use the first of january as their starting point. If they joined it during the year, then we'll use the exact date.

In [16]:
%%sql
-- Days active in 2025 before churning, split by pre-2025 vs 2025 joiners
SELECT
    CASE WHEN join_date < '2025-01-01' THEN 'Pre-2025 joiner' ELSE '2025 joiner' END AS cohort,
    COUNT(*)                                                                          AS churned_users,
    ROUND(AVG(churn_date::DATE - GREATEST(join_date::DATE, '2025-01-01'::DATE)), 0)  AS avg_days_before_churn,
    MIN(churn_date::DATE - GREATEST(join_date::DATE, '2025-01-01'::DATE))            AS min_days,
    MAX(churn_date::DATE - GREATEST(join_date::DATE, '2025-01-01'::DATE))            AS max_days,
    PERCENTILE_CONT(0.5) WITHIN GROUP (
        ORDER BY churn_date::DATE - GREATEST(join_date::DATE, '2025-01-01'::DATE)
    )                                                                                 AS median_days
FROM users
WHERE is_churned = 1
GROUP BY cohort;

Running query in 'duckdb:///:memory:'

cohort,churned_users,avg_days_before_churn,min_days,max_days,median_days
2025 joiner,5176,92.0,0,364,69.0
Pre-2025 joiner,14856,183.0,0,364,184.0


So, we see that, out of those that joined during 2025, the average was 92 days, while the median was 69. There are also users who left the same day they joined, which likely means they signed up accidentally or changed their minds right away, which might be worth investigating further in the future.

Next, we can see from which tier the users churned.

In [17]:
%%sql
-- Find each churned user's most recent subscription tier before deletion
WITH last_tier AS (
    SELECT user_id, action AS tier
    FROM (
        SELECT
            user_id,
            action,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) AS rn
        FROM subscriptions
        WHERE action IN ('free', 'plus', 'gold')
    ) ranked
    WHERE rn = 1
)
SELECT
    lt.tier                                                       AS tier_at_churn,
    COUNT(*)                                                      AS churned_users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_of_churned
FROM users u
JOIN last_tier lt ON u.user_id = lt.user_id
WHERE u.is_churned = 1
GROUP BY lt.tier
ORDER BY churned_users DESC;

Running query in 'duckdb:///:memory:'

tier_at_churn,churned_users,pct_of_churned
free,11392,56.9
plus,5283,26.4
gold,3357,16.8


lSo, we see that it is approximately the same as the general distribution of tiers, meaning there wasn't any rule to it. We can look into tiers some more - let's start with the distribution at the end of our study period.

In [18]:
%%sql
WITH latest_tier AS (
    SELECT user_id, action AS tier
    FROM (
        SELECT
            user_id,
            action,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) AS rn
        FROM subscriptions
        WHERE action IN ('free', 'plus', 'gold')
    ) ranked
    WHERE rn = 1
)
SELECT
    tier,
    COUNT(*)                                                      AS users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_of_total
FROM latest_tier
GROUP BY tier
ORDER BY users DESC;

Running query in 'duckdb:///:memory:'

tier,users,pct_of_total
free,56365,56.4
plus,26357,26.4
gold,17278,17.3


Next, it is worth finding out how many times did users switch tiers. Those who switched a higher number of times might be worth investigating additionally in the future.

In [19]:
%%sql
-- Each row in subscriptions with a tier action is a tier event.
-- Subtracting 1 from the count gives the number of switches (first event = initial state).
SELECT
    n_switches,
    COUNT(*)                                                      AS users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_of_total
FROM (
    SELECT
        user_id,
        COUNT(*) - 1                                              AS n_switches
    FROM subscriptions
    WHERE action IN ('free', 'plus', 'gold')
    GROUP BY user_id
) t
GROUP BY n_switches
ORDER BY n_switches;

Running query in 'duckdb:///:memory:'

n_switches,users,pct_of_total
0,69880,69.9
1,20064,20.1
2,8041,8.0
3,2015,2.0


Next, we can see who upgraded and who downgraded, and how often?

In [20]:
%%sql
-- Use LAG to compare each tier event to the previous one for the same user
WITH tier_changes AS (
    SELECT
        user_id,
        date,
        action                                                    AS new_tier,
        LAG(action) OVER (PARTITION BY user_id ORDER BY date)    AS prev_tier
    FROM subscriptions
    WHERE action IN ('free', 'plus', 'gold')
),
classified AS (
    SELECT
        prev_tier || ' → ' || new_tier                           AS transition,
        CASE
            WHEN (prev_tier = 'free'  AND new_tier IN ('plus', 'gold'))
              OR (prev_tier = 'plus'  AND new_tier = 'gold')      THEN 'Upgrade'
            WHEN (prev_tier = 'gold'  AND new_tier IN ('plus', 'free'))
              OR (prev_tier = 'plus'  AND new_tier = 'free')      THEN 'Downgrade'
        END                                                       AS change_type
    FROM tier_changes
    WHERE prev_tier IS NOT NULL
)
SELECT
    change_type,
    transition,
    COUNT(*)                                                      AS occurrences
FROM classified
GROUP BY change_type, transition
ORDER BY change_type, occurrences DESC;

Running query in 'duckdb:///:memory:'

change_type,transition,occurrences
Downgrade,plus → free,5819
Downgrade,gold → plus,3664
Downgrade,gold → free,3655
Upgrade,free → plus,11779
Upgrade,free → gold,11593
Upgrade,plus → gold,5646
None,free → free,29
None,gold → gold,4
None,plus → plus,2


Here we see that some kind of error slipped by in the dataset generation, which should be touched up in the future - there are some who kept the same class, but are marked as a transition. Other than that, upgrades seem to have been more common overall, with close to 30k upgrades and around 13k downgrades.

Finally, we can have a look at the actual activity. We can start off by creating a conversion funnel - how do likes translate into matches, and how they translate into messages?

In [21]:
%%sql
-- Platform-wide: how much of each action turns into the next?
SELECT
    SUM(likes)                                                    AS total_likes,
    SUM(matches)                                                  AS total_matches,
    SUM(messages_sent)                                            AS total_messages_sent,
    ROUND(SUM(matches)       * 100.0 / NULLIF(SUM(likes),   0), 2) AS like_to_match_pct,
    ROUND(SUM(messages_sent) * 100.0 / NULLIF(SUM(matches), 0), 2) AS match_to_message_pct
FROM activity;

Running query in 'duckdb:///:memory:'

total_likes,total_matches,total_messages_sent,like_to_match_pct,match_to_message_pct
220939037,11709703,17580516,5.3,150.14


So, expectedly, likes don't always lead to matches, they do in around 5% of cases, but each match seems to correspond to a message and a half - which means that on average, it consists of short interactions.

Next, we can see how the activity of different tiers occurred.

In [22]:
%%sql
-- Join users to their most recent tier, then aggregate their total activity
WITH latest_tier AS (
    SELECT user_id, action AS tier
    FROM (
        SELECT
            user_id, action,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) AS rn
        FROM subscriptions
        WHERE action IN ('free', 'plus', 'gold')
    ) ranked
    WHERE rn = 1
),
user_totals AS (
    SELECT
        user_id,
        SUM(likes)           AS total_likes,
        SUM(dislikes)        AS total_dislikes,
        SUM(matches)         AS total_matches,
        SUM(messages_sent)   AS total_msgs_sent,
        COUNT(DISTINCT date) AS active_days
    FROM activity
    GROUP BY user_id
)
SELECT
    lt.tier,
    COUNT(*)                                                      AS users,
    ROUND(AVG(ut.active_days),      0)                           AS avg_active_days,
    ROUND(AVG(ut.total_likes),      0)                           AS avg_total_likes,
    ROUND(AVG(ut.total_matches),    0)                           AS avg_total_matches,
    ROUND(AVG(ut.total_msgs_sent),  0)                           AS avg_msgs_sent,
    ROUND(AVG(ut.total_matches * 100.0 / NULLIF(ut.total_likes, 0)), 2) AS match_rate_pct
FROM latest_tier lt
JOIN user_totals ut ON lt.user_id = ut.user_id
GROUP BY lt.tier
ORDER BY
CASE
WHEN lt.tier = 'free' THEN 1
WHEN lt.tier = 'plus' THEN 2
WHEN lt.tier = 'gold' THEN 3
END ASC;

Running query in 'duckdb:///:memory:'

tier,users,avg_active_days,avg_total_likes,avg_total_matches,avg_msgs_sent,match_rate_pct
free,55018,25.0,565.0,27.0,30.0,4.12
plus,26120,51.0,4157.0,210.0,263.0,4.98
gold,17081,57.0,4758.0,279.0,531.0,5.61


So, we see that gold users to get the best match rate percent, and that they are much more active.
Next, let's see whether there is a difference in activity between churned and active users prior to churning.

In [23]:
%%sql
WITH user_totals AS (
    SELECT
        user_id,
        SUM(likes)           AS total_likes,
        SUM(matches)         AS total_matches,
        SUM(messages_sent)   AS total_msgs_sent,
        COUNT(DISTINCT date) AS active_days
    FROM activity
    GROUP BY user_id
)
SELECT
    CASE WHEN u.is_churned = 1 THEN 'Churned' ELSE 'Active' END  AS user_status,
    COUNT(*)                                                       AS users,
    ROUND(AVG(ut.active_days),     0)                             AS avg_active_days,
    ROUND(AVG(ut.total_likes),     0)                             AS avg_total_likes,
    ROUND(AVG(ut.total_matches),   0)                             AS avg_total_matches,
    ROUND(AVG(ut.total_msgs_sent), 0)                             AS avg_msgs_sent
FROM users u
JOIN user_totals ut ON u.user_id = ut.user_id
GROUP BY user_status;

Running query in 'duckdb:///:memory:'

user_status,users,avg_active_days,avg_total_likes,avg_total_matches,avg_msgs_sent
Churned,18982,22.0,1322.0,70.0,105.0
Active,79237,42.0,2472.0,131.0,197.0


Indeed, the churned users received fewer matches and used the platform less. We can investigate this further by tier - it is possible that this is not the case if we compare only the free users.


In [24]:
%%sql
WITH latest_tier AS (
    SELECT user_id, action AS tier
    FROM (
        SELECT
            user_id, action,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) AS rn
        FROM subscriptions
        WHERE action IN ('free', 'plus', 'gold')
    ) ranked
    WHERE rn = 1
),
user_totals AS (
    SELECT
        user_id,
        SUM(likes)           AS total_likes,
        SUM(matches)         AS total_matches,
        SUM(messages_sent)   AS total_msgs_sent,
        COUNT(DISTINCT date) AS active_days
    FROM activity
    GROUP BY user_id
)
SELECT
    CASE WHEN u.is_churned = 1 THEN 'Churned' ELSE 'Active' END  AS user_status,
    COUNT(*)                                                       AS users,
    ROUND(AVG(ut.active_days),     0)                             AS avg_active_days,
    ROUND(AVG(ut.total_likes),     0)                             AS avg_total_likes,
    ROUND(AVG(ut.total_matches),   0)                             AS avg_total_matches,
    ROUND(AVG(ut.total_msgs_sent), 0)                             AS avg_msgs_sent
FROM users u
JOIN user_totals ut ON u.user_id = ut.user_id
JOIN latest_tier lt ON u.user_id = lt.user_id
WHERE lt.tier = 'free'
GROUP BY user_status;

Running query in 'duckdb:///:memory:'

user_status,users,avg_active_days,avg_total_likes,avg_total_matches,avg_msgs_sent
Active,44393,28.0,621.0,29.0,33.0
Churned,10611,15.0,330.0,15.0,17.0


Even amongst the free users, it seems that activity predicts churn. However, this is overall activity, so it might be worth to have a look at only the three months prior to churning/end of study period.

In [25]:
%%sql
WITH latest_tier AS (
    SELECT user_id, action AS tier
    FROM (
        SELECT
            user_id, action,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) AS rn
        FROM subscriptions
        WHERE action IN ('free', 'plus', 'gold')
    ) ranked
    WHERE rn = 1
),
user_window AS (
    SELECT
        user_id,
        is_churned,
        CASE WHEN is_churned = 1 THEN churn_date::DATE
             ELSE '2025-12-31'::DATE
        END                                                       AS window_end
    FROM users
),
user_totals AS (
    SELECT
        a.user_id,
        SUM(a.likes)           AS total_likes,
        SUM(a.matches)         AS total_matches,
        SUM(a.messages_sent)   AS total_msgs_sent,
        COUNT(DISTINCT a.date) AS active_days
    FROM activity a
    JOIN user_window uw ON a.user_id = uw.user_id
    WHERE a.date >= uw.window_end - INTERVAL '3 months'
      AND a.date <= uw.window_end
    GROUP BY a.user_id
)
SELECT
    CASE WHEN u.is_churned = 1 THEN 'Churned' ELSE 'Active' END  AS user_status,
    COUNT(*)                                                       AS users,
    ROUND(AVG(ut.active_days),                                         0) AS avg_active_days,
    ROUND(AVG(ut.total_likes),                                         0) AS avg_total_likes,
    ROUND(AVG(ut.total_matches),                                       0) AS avg_total_matches,
    ROUND(AVG(ut.total_msgs_sent),                                     0) AS avg_msgs_sent,
    ROUND(AVG(ut.total_matches / NULLIF(ut.active_days,   0)::FLOAT),  3) AS avg_matches_per_day,
    ROUND(AVG(ut.total_matches / NULLIF(ut.total_likes,   0)::FLOAT),  3) AS avg_matches_per_like,
    ROUND(AVG(ut.total_msgs_sent / NULLIF(ut.total_matches, 0)::FLOAT),3) AS avg_msgs_per_match
FROM users u
JOIN user_totals ut ON u.user_id = ut.user_id
JOIN latest_tier lt ON u.user_id = lt.user_id
WHERE lt.tier = 'free'
GROUP BY user_status;

Running query in 'duckdb:///:memory:'

user_status,users,avg_active_days,avg_total_likes,avg_total_matches,avg_msgs_sent,avg_matches_per_day,avg_matches_per_like,avg_msgs_per_match
Active,44336,7.0,108.0,5.0,4.0,0.595,0.04,0.548
Churned,10607,7.0,129.0,6.0,5.0,0.669,0.041,0.571


Here we see that our churners actually had more success with their likes and matches. This maybe means that they found what they were looking for and left the site for that reason!

Next, we can find our most active users - maybe there is something interesting about them!

In [26]:
%%sql
WITH latest_tier AS (
    SELECT user_id, action AS tier
    FROM (
        SELECT
            user_id, action,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY date DESC) AS rn
        FROM subscriptions
        WHERE action IN ('free', 'plus', 'gold')
    ) ranked
    WHERE rn = 1
)
SELECT
    a.user_id,
    u.gender,
    u.age,
    u.country,
    u.work_status,
    lt.tier,
    SUM(a.likes)           AS total_likes,
    SUM(a.matches)         AS total_matches,
    SUM(a.messages_sent)   AS total_msgs_sent,
    COUNT(DISTINCT a.date) AS active_days
FROM activity a
JOIN users u ON a.user_id = u.user_id
JOIN latest_tier lt ON a.user_id = lt.user_id
GROUP BY a.user_id, u.gender, u.age, u.country, u.work_status, lt.tier
ORDER BY total_likes DESC
LIMIT 20;

Running query in 'duckdb:///:memory:'

user_id,gender,age,country,work_status,tier,total_likes,total_matches,total_msgs_sent,active_days
70868,Male,29,Spain,Unemployed,gold,20597,1274,2595,140
11815,Male,23,United Kingdom,Student,plus,19976,1163,2292,136
54880,Male,26,France,Employed,gold,19911,1194,2306,139
44901,Male,26,United Kingdom,Student,gold,19728,1202,2426,131
65637,Male,20,Germany,Employed,gold,19116,1146,2238,121
61354,Male,26,France,Employed,gold,19109,1138,2283,133
42913,Male,25,Germany,Student,gold,19001,1164,2266,143
65403,Male,28,Spain,Employed,gold,18885,1083,2107,133
70411,Male,27,France,Unemployed,gold,18786,1146,2179,126
59199,Male,26,Germany,Employed,gold,18730,1110,2253,141


So, all of our top 20 users are men in their 20s,most of them with a gold tier, although one plus tier subscriber also got into the list - maybe he used to be a gold tier but then switched. Let's check.

In [27]:
%%sql
SELECT
    date,
    action
FROM subscriptions
WHERE user_id = 11815
ORDER BY date;

Running query in 'duckdb:///:memory:'

date,action
2025-01-01 00:00:00,gold
2025-12-14 00:00:00,plus


Yes, as expected, se also used to be a gold-tier user, but switched to plus towards the end of the year, which is why his subscription tier is plus in the summary table above.

Finally, we can apply some more advanced SQL to see some more interesting metrics. For one, let's see the monthly cohort churn.

In [28]:
%%sql
WITH cohorts AS (
    SELECT
        user_id,
        STRFTIME(join_date, '%Y-%m')                              AS cohort_month,
        is_churned
    FROM users
    WHERE join_date >= '2025-01-01'
)
SELECT
    cohort_month,
    COUNT(*)                                                      AS new_users,
    SUM(is_churned)                                               AS churned_by_year_end,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 1)                 AS churn_rate_pct
FROM cohorts
GROUP BY cohort_month
ORDER BY cohort_month;

Running query in 'duckdb:///:memory:'

cohort_month,new_users,churned_by_year_end,churn_rate_pct
2025-01,2167,449,20.7
2025-02,1926,401,20.8
2025-03,2099,438,20.9
2025-04,2080,447,21.5
2025-05,2152,451,21.0
2025-06,2118,437,20.6
2025-07,2099,417,19.9
2025-08,2142,423,19.7
2025-09,2067,439,21.2
2025-10,2045,420,20.5


So, we see that our churn is even across the dataset - again, this is just a consequence of the artificial dataset. Next, let's have a look at how long did users use the platform before leaving.

In [29]:
%%sql
-- Days active in 2025 before churn = churn_date - max(join_date, 2025-01-01)
WITH days_active AS (
    SELECT
        user_id,
        churn_date::DATE - GREATEST(join_date::DATE, '2025-01-01'::DATE) AS days
    FROM users
    WHERE is_churned = 1
)
SELECT
    COUNT(*)                                                      AS total_churned,
    SUM(CASE WHEN days <= 7  THEN 1 ELSE 0 END)                  AS within_1_week,
    SUM(CASE WHEN days <= 30 THEN 1 ELSE 0 END)                  AS within_30_days,
    SUM(CASE WHEN days <= 90 THEN 1 ELSE 0 END)                  AS within_90_days,
    SUM(CASE WHEN days >  90 THEN 1 ELSE 0 END)                  AS after_90_days,
    ROUND(SUM(CASE WHEN days <= 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_quick_churn
FROM days_active;

Running query in 'duckdb:///:memory:'

total_churned,within_1_week,within_30_days,within_90_days,after_90_days,pct_quick_churn
20032,851,2683,6613,13419,13.4


So, most of our churn happens after 90 days, meaning that we do not have a quick churn issue - only around 13% happens within 30 days. Next, let's look at who our best performing users are (getting the most matches per like)

In [30]:
%%sql
-- QUALIFY filters on window function results without a subquery wrapper
WITH user_stats AS (
    SELECT
        a.user_id,
        u.country,
        u.gender,
        u.age,
        SUM(a.likes)                                              AS total_likes,
        SUM(a.matches)                                            AS total_matches,
        ROUND(SUM(a.matches) * 100.0 / NULLIF(SUM(a.likes), 0), 2) AS match_rate_pct
    FROM activity a
    JOIN users u ON a.user_id = u.user_id
    GROUP BY a.user_id, u.country, u.gender, u.age
    HAVING SUM(a.likes) > 200  -- exclude users with too little data for a meaningful rate
)
SELECT
    country,
    user_id,
    gender,
    age,
    total_likes,
    total_matches,
    match_rate_pct,
    RANK() OVER (PARTITION BY country ORDER BY match_rate_pct DESC) AS rank_in_country
FROM user_stats
QUALIFY RANK() OVER (PARTITION BY country ORDER BY match_rate_pct DESC) <= 5
ORDER BY country, rank_in_country;

Running query in 'duckdb:///:memory:'

country,user_id,gender,age,total_likes,total_matches,match_rate_pct,rank_in_country
France,93926,Female,23,298,30,10.07,1
France,87197,Male,21,225,21,9.33,2
France,56295,Male,35,259,24,9.27,3
France,97728,Female,23,545,48,8.81,4
France,79529,Male,21,743,65,8.75,5
Germany,80537,Male,30,209,23,11.0,1
Germany,15152,Male,48,225,22,9.78,2
Germany,78503,Male,44,329,31,9.42,3
Germany,91081,Male,28,215,20,9.3,4
Germany,97245,Male,30,309,28,9.06,5


In spite of the general population being mostly male, it seems that both women and non-binary persons are successful enough to get into the lists. This includes those in their 20s and 30s alike.
We can also have a look at the other side of the spectrum - who are the most active, least successful users?

In [31]:
%%sql
WITH user_stats AS (
    SELECT
        a.user_id,
        u.gender,
        u.country,
        SUM(a.likes)                                              AS total_likes,
        SUM(a.matches)                                            AS total_matches,
        ROUND(SUM(a.matches) * 100.0 / NULLIF(SUM(a.likes), 0), 2) AS match_rate_pct
    FROM activity a
    JOIN users u ON a.user_id = u.user_id
    GROUP BY a.user_id, u.gender, u.country
    HAVING SUM(a.likes) > 100
),
thresholds AS (
    SELECT
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY total_likes)   AS like_p75,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY match_rate_pct) AS match_p25
    FROM user_stats
)
SELECT
    us.gender,
    us.country,
    COUNT(*)                                                      AS flagged_users,
    ROUND(AVG(us.total_likes),     0)                             AS avg_likes,
    ROUND(AVG(us.match_rate_pct),  2)                             AS avg_match_rate_pct
FROM user_stats us, thresholds t
WHERE us.total_likes     >= t.like_p75
  AND us.match_rate_pct  <= t.match_p25
GROUP BY us.gender, us.country
ORDER BY flagged_users DESC;

Running query in 'duckdb:///:memory:'

gender,country,flagged_users,avg_likes,avg_match_rate_pct
Male,Spain,1,5260.0,3.78


It turns out there is only one such user, being in the top 25% of liking but in the bottom 25% of matches. The lack of such users indicates the algorithm functions as intended, and that this person simply has little luck.

Ending on that, this notebook demonstrates a variety of SQL skills that could be useful in a similar project, such as:


| Topic                  | Techniques used                                                          |
|------------------------|--------------------------------------------------------------------------|
| Demographics           | `GROUP BY`, `COUNT`, `AVG`, window `%` share                             |
| Churn analysis         | `CASE WHEN` bucketing, `JOIN`, `PERCENTILE_CONT`                         |
| Subscription behaviour | `LAG()`, CTE chains, conditional aggregation                             |
| Engagement funnel      | `NULLIF`, multi-level conversion rates                                   |
| Advanced               | `RANK()`, `QUALIFY`, correlated subqueries, `PERCENTILE_CONT` thresholds |

The data is simulated, so the numbers are by design not too interesting, but the queries could literally be transferred to any real-world user behaviour dataset.